# BISINDO Static Sign Language Classifier
Extracts hand landmarks from images using MediaPipe (both hands, 126 features), normalizes them relative to each hand's wrist, and trains a Dense classifier for letters A–Z (folders 0–25).

In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
print('MediaPipe:', mp.__version__)

import tensorflow as tf
print('TensorFlow:', tf.__version__)

In [ ]:
DATA_DIR = './'  # Folders 0-25 are in the same dir as this notebook
LETTERS = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
EMPTY_HAND = np.zeros(63)  # Placeholder when a hand is not detected

def normalize_landmarks(landmarks):
    """Normalize landmarks relative to wrist (index 0) and hand scale."""
    wrist = landmarks[0].copy()
    landmarks = landmarks - wrist
    scale = np.linalg.norm(landmarks[9])  # middle finger MCP
    if scale > 0:
        landmarks = landmarks / scale
    return landmarks

def extract_landmarks_from_image(image_path, hands):
    """
    Returns a (126,) array: [hand_0 (63,) | hand_1 (63,)]
    Missing hands are filled with zeros.
    Returns None if NO hand is detected at all.
    """
    img = cv2.imread(image_path)
    if img is None:
        return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    if not results.multi_hand_landmarks:
        return None

    hand_features = [EMPTY_HAND.copy(), EMPTY_HAND.copy()]  # slot 0 & 1

    for i, hand_lm in enumerate(results.multi_hand_landmarks[:2]):
        raw = np.array([[lm.x, lm.y, lm.z] for lm in hand_lm.landmark])  # (21, 3)
        normalized = normalize_landmarks(raw)
        hand_features[i] = normalized.flatten()  # (63,)

    return np.concatenate(hand_features)  # (126,)

X, y = [], []

with mp_hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.3) as hands:
    for class_idx in range(26):
        folder = os.path.join(DATA_DIR, str(class_idx))
        if not os.path.isdir(folder):
            print(f'  Skipping missing folder: {folder}')
            continue
        images = [f for f in os.listdir(folder) if f.lower().endswith('.jpg')]
        count = 0
        for img_file in images:
            path = os.path.join(folder, img_file)
            features = extract_landmarks_from_image(path, hands)
            if features is not None:
                X.append(features)
                y.append(class_idx)
                count += 1
        print(f'  Class {LETTERS[class_idx]} (folder {class_idx}): {count}/{len(images)} detected')

X = np.array(X)
y = np.array(y)
print(f'\nTotal samples: {len(X)}, Feature shape: {X.shape}')

In [ ]:
lb = LabelBinarizer()
y_onehot = lb.fit_transform(y)  # (N, 26)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
num_classes = y_onehot.shape[1]

model = Sequential([
    Dense(256, activation='relu', input_shape=(126,)),
    BatchNormalization(),
    Dropout(0.4),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1)
]

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=callbacks
)

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy: {acc:.4f} ({acc*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
model.save('bisindo_static_model.h5')
print('Model saved to bisindo_static_model.h5')